# Lab 10 -- Decision Trees

Compared to last week, this is a very simple lab <span style="font-size:20pt;">😃</span> You'll have fun programming!

You will implement the **Classification and Regression Tree (CART)** algorithm from scratch.

The lab is broken down into the following pieces:

* Regression Criterion
* Creating Splits
* Buiding a Tree
* Making a prediction


# Decision trees for Regression
## Exercise 1 -- Download and load the dataset

We will be using the usual Boston Housing dataset, which is available to download from ECLASS

* Download the file
* Read it and separate the target variable from the features.
* Make a 80/10/10 train/validation/test split

In [475]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [476]:
housing_names = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
data = pd.read_table("housing.txt", names=housing_names, sep=r'\s+')

The target variable will be as usual `MEDV`. Use the rest as features.

In [477]:
X = data.iloc[:, :-1].values
y = data["MEDV"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5)

## Exercise 2 -- Optimization Criterion

For regression, a simple criterion to optimize is to minimize the sum of squared errors for a given region. This is, for all datapoints in a region with size, we minimize:

$$\sum_{i=1}^N(y_i - \hat{y})^2$$

where $N$ is the number of datapoits in the region and $\hat{y}$ is the mean value of the region for the target variable. 

Implement such a function using the description below.

Please, don't use an existing implementation, refer to the [book](https://www.statlearning.com/s/ISLRSeventhPrinting.pdf), and if you need help, ask questions!

In [478]:
def regression_criterion(region):
    """
    Implements the sum of squared error criterion in a region
    
    Parameters
    ----------
    region : ndarray
        Array of shape (N,) containing the values of the target values 
        for N datapoints in the training set.
    
    Returns
    -------
    float
        The sum of squared error
        
    Note
    ----
    The error for an empty region should be infinity (use: float("inf"))
    This avoids creating empty regions
    """
    if region.size == 0: 
        return float("inf")
        
    return np.sum((region - np.mean(region))**2)

In [479]:
# test your code
rng = np.random.default_rng(0)
print(regression_criterion(rng.random(size=40)))
print(regression_criterion(np.ones(10)))
print(regression_criterion(np.zeros(10)))
print(regression_criterion(np.array([])))

3.6200679838629544
0.0
0.0
inf


## Exercise 3 -- Make a split

In [480]:
def split_region(region, feature_index, tau):
    """
    Given a region, splits it based on the feature indicated by
    `feature_index`, the region will be split in two, where
    one side will contain all points with the feature with values 
    lower than `tau`, and the other split will contain the 
    remaining datapoints.
    
    Parameters
    ----------
    region : array of size (n_samples, n_features)
        a partition of the dataset (or the full dataset) to be split
    feature_index : int
        the index of the feature (column of the region array) used to make this partition
    tau : float
        The threshold used to make this partition
        
    Return
    ------
    left_partition : array
        indices of the datapoints in `region` where feature < `tau`
    right_partition : array
        indices of the datapoints in `region` where feature >= `tau` 
    """
    left_partition = region[region[:, feature_index] < tau]
    right_partition = region[region[:, feature_index] >= tau]

    return left_partition, right_partition

## Exercise 4 -- Find the best split

The strategy is quite simple (as well as inefficient), but it helps to reinforce the concepts.
We are going to use a greedy, exhaustive algorithm to select splits, selecting the `feature_index` and the `tau` that minimizes the Regression Criterion

In [481]:
def get_split(X, y):
    """
    Given a dataset (full or partial), splits it on the feature of that minimizes the sum of squared error
    
    Parameters
    ----------
    X : array (n_samples, n_features)
        features 
    y : array (n_samples, )
        labels
    
    Returns
    -------
    decision : dictionary
        keys are:
        * 'feature_index' -> an integer that indicates the feature (column) of `X` on which the data is split
        * 'tau' -> the threshold used to make the split
        * 'left_region' -> array of indices where the `feature_index`th feature of X is lower than `tau`
        * 'right_region' -> indices not in `low_region`
    """
    decision = dict()
    best_error = np.inf
    for feature_index in range(X_train.shape[1]):
        min = X[:, feature_index].min()
        max = X[:, feature_index].max()
        if min == max:
            continue
        step = (max - min) / X_train.shape[0]
        for tau in np.arange(min, max, step):
            left_partition, right_partition = split_region(X, feature_index, tau)

            left_error = regression_criterion(y[X[:, feature_index] < tau])
            right_error = regression_criterion(y[X[:, feature_index] >= tau])
            total_error = left_error + right_error

            if total_error < best_error:
                best_error = total_error
                decision["feature_index"] = feature_index
                decision["tau"] = tau
                decision["left_region"] = np.where(X[:, feature_index] < tau)[0]
                decision["right_region"] = np.where(X[:, feature_index] >= tau)[0]

    return decision

## Exercise 5 -- Recursive Splitting

The test above is an example on how to find the root node of our decision tree. The algorithm now is a greedy search until we reach a stop criterion. To find the actual root node of our decision tree, you must provide the whole training set, not just a slice of 15 rows as the test above.

The trivial stopping criterion is to recursively grow the tree until each split contains a single point (perfect node purity). If we go that far, it normally means we are overfitting.

You will implement these criteria to stop the growth:

* A node is a leaf if:
    * It has less than `min_samples` datapoints
    * It is at the `max_depth` level from the root (each split creates a new level)
    * The criterion is `0`



In [482]:
def recursive_growth(node, min_samples, max_depth, current_depth, X, y):
    """
    Recursively grows a decision tree.
    
    Parameters
    ----------
    node : dictionary
        If the node is terminal, it contains only the "value" key, which determines the value to be used as a prediction.
        If the node is not terminal, the dictionary has the structure defined by `get_split`
    min_samples : int
        parameter for stopping criterion if a node has <= min_samples datapoints
    max_depth : int
        parameter for stopping criterion if a node belongs to this depth
    depth : int
        current distance from the root
    X : array (n_samples, n_features)
        features (full dataset)
    y : array (n_samples, )
        labels (full dataset)
    
    Notes
    -----
    To create a terminal node, a dictionary is created with a single "value" key, with a value that
    is the mean of the target variable
    
    'left' and 'right' keys are added to non-terminal nodes, which contain (possibly terminal) nodes 
    from higher levels of the tree:
    'left' corresponds to the 'left_region' key, and 'right' to the 'right_region' key
    """
    left_idx, right_idx = node["left_region"], node["right_region"]

    X_left, y_left = X[left_idx], y[left_idx]
    X_right, y_right = X[right_idx], y[right_idx]

    if current_depth >= max_depth:
        node["left"] = {"value": np.mean(y_left) if len(y_left) > 0 else 0.0}
        node["right"] = {"value": np.mean(y_right) if len(y_right) > 0 else 0.0}

        return node

    if len(y_left) <= min_samples:
        node["left"] = {"value": np.mean(y_left) if len(y_left) > 0 else 0.0}
    else:
        left_decision = get_split(X_left, y_left)
        node["left"] = recursive_growth(left_decision, min_samples, max_depth, current_depth + 1, X, y)

    if len(y_right) <= min_samples:
        node["right"] = {"value": np.mean(y_right) if len(y_right) > 0 else 0.0}
    else:
        right_decision = get_split(X_right, y_right)
        node["right"] = recursive_growth(right_decision, min_samples, max_depth, current_depth + 1, X, y)

    return node

decision = get_split(X_train, y_train)
print(decision["feature_index"], decision["tau"])
print(X_train[decision["left_region"]][:, decision["feature_index"]])
print(X_train[decision["right_region"]][:, decision["feature_index"]])

print(regression_criterion(y_train))
print(regression_criterion(y_train[decision["left_region"]]))
print(regression_criterion(y_train[decision["right_region"]]))

5 6.942210396039572
[6.718 5.872 5.928 5.56  6.726 6.279 6.481 5.597 5.536 6.849 6.552 6.49
 6.335 6.172 6.096 5.818 6.63  5.79  6.373 6.185 6.406 6.631 5.412 6.23
 6.095 6.635 6.167 6.113 6.144 6.302 5.987 6.453 6.345 6.316 5.854 5.869
 6.874 5.836 5.605 6.593 6.162 5.852 5.895 6.223 6.066 5.572 6.037 5.96
 6.649 6.421 5.404 5.857 5.92  5.713 6.092 6.266 6.108 5.887 5.362 6.674
 6.02  6.317 6.245 6.065 6.101 6.425 5.965 6.458 6.63  6.341 6.12  6.782
 6.879 5.627 6.209 5.87  5.707 6.567 6.833 6.019 5.399 5.871 5.403 6.122
 5.304 6.072 5.693 5.    6.416 5.957 6.273 6.312 6.129 6.24  6.041 6.377
 6.516 5.813 6.438 5.036 6.301 5.613 6.398 6.38  5.868 6.762 5.963 6.096
 6.8   5.885 6.315 6.728 4.138 6.749 5.837 5.972 6.459 5.936 6.383 6.75
 5.983 6.393 6.127 5.875 6.513 6.618 5.936 5.926 4.88  6.701 6.142 6.086
 5.803 6.826 6.211 5.981 6.372 5.631 5.787 5.565 4.903 6.794 4.906 5.468
 6.629 6.405 6.727 6.208 6.014 5.277 6.727 6.842 6.781 6.137 6.415 6.144
 5.569 5.093 6.739 6.297 6.43  6.34

Below we provide code to visualise the generated tree!

In [483]:
def print_tree(node, depth):
    if "value" in node:
        print(node["value"])
        return

    if "left" in node:
        print_tree(node["left"], depth + 1)

    if "right" in node:
        print_tree(node["right"], depth + 1)

In [484]:
root = recursive_growth(get_split(X_train, y_train), 10, 7, 0, X_train, y_train)

print_tree(root, 0)

22.34909090909091
24.0
21.761290322580646
28.0625
23.57
20.15
21.4
25.342857142857145
16.9
23.042857142857148
18.1
16.28
20.0
21.625
19.075000000000003
27.283333333333335
16.700000000000003
23.683333333333334
19.96666666666667
19.075000000000003
27.325
21.418518518518518
22.333333333333332
21.900000000000002
20.46
27.950000000000003
24.633333333333336
21.266666666666666
22.34347826086957
18.771428571428572
27.46
22.46666666666667
19.78
18.3
23.979999999999997
21.133333333333333
22.516666666666666
24.549999999999997
20.428571428571427
22.675
16.816666666666666
22.75714285714286
20.52
22.95555555555556
21.7


## Exercise 6 -- Make a Prediction
Use the a node to predict the class of a compatible dataset

In [485]:
def predict_sample(node, sample):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    sample : array of size (n_features,)
        a sample datapoint
    """
    pass
        
def predict(node, X):
    """
    Makes a prediction based on the decision tree defined by `node`
    
    Parameters
    ----------
    node : dictionary
        A node created one of the methods above
    X : array of size (n_samples, n_features)
        n_samples predictions will be made
    """
    pass

Now use the functions defined above to calculate the RMSE of the validation set. 
* Try first with `min_samples=20` and `max_depth=6` (for this values you should get a validation RMSE of ~8.8)

Then, experiment with different values for the stopping criteria.

In [486]:
# calculate root mean squared error with numpy
def root_mean_squared_error(y_true, y_pred):
    """
    Calculates the root mean squared error between two arrays
    
    Parameters
    ----------
    y_true : array of size (n_samples,)
        true labels
    y_pred : array of size (n_samples,)
        predicted labels
    """
    return np.sqrt(np.mean((y_true - y_pred)**2))

In [487]:
root = get_split(X_train, y_train)
min_samples = 20
max_depth = 6
recursive_growth(root, min_samples, max_depth, 1, X_train, y_train)
train_mse = root_mean_squared_error(y_train, predict(root, X_train))
test_mse = root_mean_squared_error(y_test, predict(root, X_test))

print(f'Train MSE : {train_mse}')
print(f'Test MSE : {test_mse}')

TypeError: unsupported operand type(s) for -: 'float' and 'NoneType'

## Just to see how much we can improve this naive decision tree!

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
reg_train_mse = root_mean_squared_error(y_train, reg.predict(X_train))
reg_test_mse = root_mean_squared_error(y_test, reg.predict(X_test))
print(f'Regression Train MSE : {reg_train_mse}')
print(f'Regression Test MSE : {reg_test_mse}')

Regression Train MSE : 4.396188144698282
Regression Test MSE : 5.931426809490862


# Decision trees for Classification
You will implement decision trees for classification using the Gini index as the splitting criterion. You’ll build the tree recursively, selecting splits that minimize Gini impurity and classifying samples based on majority class in each leaf. A good dataset to start with is the Iris dataset, which is small, well-labeled, and available directly via `sklearn.datasets.load_iris().`

In [ ]:
# Load the Iris dataset
data = load_iris()
X = data.data
y = data.target

We will focus only on binary classification today!

In [ ]:
X = X[y != 2]
y = y[y != 2]

In [ ]:
# split your data into training, validation and test sets!

### Feel free to use the same code as for regression, but change the criterion and the prediction function. You can also implement a new one if you want to!

In [ ]:
# you will test 3 values for min_samples_split: 2, 4, 6
# Remember that this sets the minimum number of samples required in a node to be eligible for splitting. 
# These values are good for small datasets like Iris, but you can try other values for larger datasets to not make the tree too deep.

# your code goes here

Use accuracy to find the best split. Don't import it from sklearn, calculate it yourself, it's a one-liner ;)

In [ ]:
## Lastly, evaluate your model on the test set and print the accuracy score

# your code goes here